# MVA Hackathon 2026

This notebook runs the public, auditable pipeline against the gated data in Colab. It downloads only the VCF, index, and phenotype document first. Do not save patient-derived files to GitHub or share them outside the authorized workspace.

In [ ]:
!git clone -b codex/mva-hackathon-2026 https://github.com/liamc225/mva-hackathon-2026.git
%cd mva-hackathon-2026
!pip -q install -r requirements.txt

## Authenticate and request the small analysis files

Run `login()` interactively with your Hugging Face token only after you have accepted the gated-dataset terms in the Hugging Face UI. The access request is a separate human/legal step.

In [ ]:
from huggingface_hub import login, hf_hub_download, list_repo_files
import subprocess
from pathlib import Path

# Interactive prompt; the token is not saved in this repository.
login()
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
Path('runtime').mkdir(exist_ok=True)
repo_id = 'SageBio/mva-hackathon-2026-data'
repo_files = list_repo_files(repo_id, repo_type='dataset')
vcf_name = next(name for name in repo_files if name.endswith('.vcf.gz') and not name.endswith('.vcf.gz.tbi'))
vcf_index_name = vcf_name + '.tbi'
phenotype_name = next(name for name in repo_files if name.lower().endswith('.docx'))
vcf_path = Path(hf_hub_download(repo_id, filename=vcf_name, repo_type='dataset', local_dir=str(data_dir)))
hf_hub_download(repo_id, filename=vcf_index_name, repo_type='dataset', local_dir=str(data_dir))
phenotype_path = Path(hf_hub_download(repo_id, filename=phenotype_name, repo_type='dataset', local_dir=str(data_dir)))
print(vcf_path, phenotype_path)

In [ ]:
subprocess.run(['python', 'scripts/inspect_dataset.py', '--vcf', str(vcf_path)], check=True)
subprocess.run(['python', 'scripts/extract_phenotype.py', '--docx', str(phenotype_path)], check=True)
print(Path('runtime/phenotype.txt').read_text())

## Local panel and coding-aware ranking

The challenge VCF has no ANN/CSQ consequence fields. First bound the local review to a public GRCh38 MVA gene panel, then use GENCODE v47 and public Ensembl reference sequence locally. The private VCF and derived table stay in Colab.

In [ ]:
subprocess.run(['python', 'scripts/rank_mva_panel.py', '--vcf', str(vcf_path), '--max-rows', '100'], check=True)

import requests
gtf_url = 'https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_47/gencode.v47.basic.annotation.gtf.gz'
gtf_path = Path('runtime/gencode.v47.basic.annotation.gtf.gz')
if not gtf_path.exists():
    gtf_path.write_bytes(requests.get(gtf_url, timeout=120).content)
!gzip -dc runtime/gencode.v47.basic.annotation.gtf.gz > runtime/gencode.v47.basic.annotation.gtf
subprocess.run(['python', 'scripts/annotate_coding.py', '--vcf', str(vcf_path), '--gtf', 'runtime/gencode.v47.basic.annotation.gtf', '--out', 'runtime/mva_coding_rank.csv', '--max-rows', '100'], check=True)